# 02 - Generate COTs

Generate chain-of-thought traces for all GSM8K test problems using the PRIMARY_MODEL (Qwen3-4B).
Also runs the no_cot baseline condition.

**Fully resumable** - caches one file per problem.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import re
from tqdm import tqdm
from lib.data import load_gsm8k, extract_predicted_answer
from lib.prompts import build_cot_messages, build_no_cot_messages

dataset = load_gsm8k()
print(f"Loaded {len(dataset)} GSM8K problems")

## Load PRIMARY_MODEL

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PRIMARY_MODEL)
print("Model loaded.")

## Phase 1: Generate COTs (thinking mode)

In [ ]:
# Check resume state
done_ids = {int(p.stem) for p in COT_CACHE.glob("*.json")}
todo = [ex for ex in dataset if ex["problem_id"] not in done_ids]
print(f"COT generation: {len(done_ids)} done, {len(todo)} remaining")

In [ ]:
sampling_cot = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc="Generating COTs"):
    chunk = todo[i:i + CHUNK_SIZE]

    # Build prompts using chat template
    prompts = []
    for ex in chunk:
        messages = build_cot_messages(ex["question"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=True
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling_cot)

    for ex, output in zip(chunk, outputs):
        full_response = output.outputs[0].text

        # Extract thinking portion (between <think> tags) and answer
        think_match = re.search(r"<think>(.*?)</think>", full_response, re.DOTALL)
        if think_match:
            cot_text = think_match.group(1).strip()
            answer_portion = full_response[think_match.end():].strip()
        else:
            # Fallback: treat everything before "The answer is" as COT
            parts = re.split(r"The answer is:?", full_response, maxsplit=1)
            cot_text = parts[0].strip()
            answer_portion = parts[1].strip() if len(parts) > 1 else full_response

        predicted = extract_predicted_answer(answer_portion)
        if predicted is None:
            predicted = extract_predicted_answer(full_response)

        result = {
            "problem_id": ex["problem_id"],
            "question": ex["question"],
            "gold_answer": ex["gold_answer"],
            "cot_text": cot_text,
            "predicted_answer": predicted,
            "full_response": full_response,
        }
        cache_path = COT_CACHE / f"{ex['problem_id']}.json"
        cache_path.write_text(json.dumps(result))

print(f"COT generation complete. Total cached: {len(list(COT_CACHE.glob('*.json')))}")

## Phase 2: No-COT baseline

Same model, direct answer only.

In [ ]:
# Check resume state for no_cot
no_cot_done = {
    int(p.stem.split("_")[1])
    for p in PREFILL_CACHE.glob("no_cot_*.json")
}
no_cot_todo = [ex for ex in dataset if ex["problem_id"] not in no_cot_done]
print(f"No-COT baseline: {len(no_cot_done)} done, {len(no_cot_todo)} remaining")

In [ ]:
sampling_short = SamplingParams(temperature=TEMPERATURE, max_tokens=64)

for i in tqdm(range(0, len(no_cot_todo), CHUNK_SIZE), desc="No-COT baseline"):
    chunk = no_cot_todo[i:i + CHUNK_SIZE]

    prompts = []
    for ex in chunk:
        messages = build_no_cot_messages(ex["question"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling_short)

    for ex, output in zip(chunk, outputs):
        generated = output.outputs[0].text.strip()
        predicted = extract_predicted_answer(generated)

        result = {
            "problem_id": ex["problem_id"],
            "condition": "no_cot",
            "model_used": PRIMARY_MODEL,
            "prefill_text": None,
            "predicted_answer": predicted,
            "gold_answer": ex["gold_answer"],
            "generated_tokens": generated,
            "error": None,
        }
        cache_path = PREFILL_CACHE / f"no_cot_{ex['problem_id']}.json"
        cache_path.write_text(json.dumps(result))

print(f"No-COT baseline complete. Total cached: {len(list(PREFILL_CACHE.glob('no_cot_*.json')))}")

## Quick validation

In [ ]:
# Quick accuracy check on COTs generated so far
correct = 0
total = 0
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    total += 1
    if r["predicted_answer"] == r["gold_answer"]:
        correct += 1

print(f"COT accuracy (normal condition): {correct}/{total} = {correct/total:.3f}")

# No-COT accuracy
correct_nc = 0
total_nc = 0
for p in PREFILL_CACHE.glob("no_cot_*.json"):
    r = json.loads(p.read_text())
    total_nc += 1
    if r["predicted_answer"] == r["gold_answer"]:
        correct_nc += 1

print(f"No-COT accuracy: {correct_nc}/{total_nc} = {correct_nc/total_nc:.3f}")
print(f"COT value: {correct/total - correct_nc/total_nc:.3f}")

In [ ]:
# Clean up
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("Model unloaded.")